# UdaPlay — Part 1: RAG Pipeline

This notebook:
1. Loads game and company data from `data/games.json`
2. Formats records into text documents
3. Embeds and stores them in a persistent ChromaDB collection
4. Demonstrates semantic search with three example queries

## 1. Setup & Environment

In [1]:
import os
import json
from dotenv import load_dotenv

load_dotenv()

OPENAI_API_KEY = os.getenv('OPENAI_API_KEY')
assert OPENAI_API_KEY is not None, 'Missing OPENAI_API_KEY in .env'

print('Environment loaded OK')

Environment loaded OK


## 2. Load Game Data from JSON

In [2]:
with open('data/games.json', 'r', encoding='utf-8') as f:
    raw_data = json.load(f)

games = raw_data['games']
companies = raw_data['companies']

print(f'Loaded {len(games)} games and {len(companies)} companies')
print(f'\nSample game: {games[0]["title"]} ({games[0]["release_date"]})')
print(f'Sample company: {companies[0]["name"]} — {companies[0]["country"]}')

Loaded 22 games and 10 companies

Sample game: Elden Ring (2022-02-25)
Sample company: Nintendo — Japan


## 3. Format Documents for Embedding

Each game and company record is converted into a plain-text document that captures all key facts. This rich text format gives the embedding model maximum context.

In [3]:
def format_game_document(game: dict) -> str:
    platforms = ', '.join(game['platforms'])
    genres = ', '.join(game['genre'])
    score = str(game['metacritic_score']) if game['metacritic_score'] else 'N/A'
    awards = '; '.join(game['awards']) if game['awards'] else 'None'
    return (
        f"Title: {game['title']}. "
        f"Developer: {game['developer']}. "
        f"Publisher: {game['publisher']}. "
        f"Release date: {game['release_date']}. "
        f"Platforms: {platforms}. "
        f"Genre: {genres}. "
        f"Metacritic score: {score}. "
        f"Awards: {awards}. "
        f"Description: {game['description']}"
    )


def format_company_document(company: dict) -> str:
    projects = '; '.join(company['current_projects'])
    series = ', '.join(company['notable_series'])
    parent = company.get('parent_company', 'Independent')
    return (
        f"Company: {company['name']}. "
        f"Country: {company['country']}. "
        f"Founded: {company['founded']}. "
        f"Headquarters: {company['headquarters']}. "
        f"Parent company: {parent}. "
        f"Current projects: {projects}. "
        f"Notable series: {series}. "
        f"Description: {company['description']}"
    )


# Build document lists
game_docs = [format_game_document(g) for g in games]
game_ids  = [g['id'] for g in games]
game_meta = [{'type': 'game', 'title': g['title'], 'developer': g['developer'],
               'release_date': g['release_date'], 'metacritic_score': str(g['metacritic_score'])}
              for g in games]

company_docs = [format_company_document(c) for c in companies]
company_ids  = [f"company-{c['id']}" for c in companies]
company_meta = [{'type': 'company', 'name': c['name'], 'country': c['country'],
                  'founded': str(c['founded'])} for c in companies]

all_docs = game_docs + company_docs
all_ids  = game_ids  + company_ids
all_meta = game_meta + company_meta

print(f'Total documents prepared: {len(all_docs)}')
print('\nSample game document:')
print(game_docs[0][:300], '...')

Total documents prepared: 32

Sample game document:
Title: Elden Ring. Developer: FromSoftware. Publisher: Bandai Namco Entertainment. Release date: 2022-02-25. Platforms: PC, PlayStation 4, PlayStation 5, Xbox One, Xbox Series X/S. Genre: Action RPG, Soulslike, Open World. Metacritic score: 96. Awards: Game of the Year 2022 - The Game Awards. Descri ...


## 4. Initialize ChromaDB & Embed Documents

In [4]:
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

# Persistent client — data survives between runs
db_client = chromadb.PersistentClient(path='./chroma_db')

embedding_fn = OpenAIEmbeddingFunction(
    api_key=OPENAI_API_KEY,
    model_name='text-embedding-3-small',
)

# Get or create the collection
collection = db_client.get_or_create_collection(
    name='games',
    embedding_function=embedding_fn,
    metadata={'hnsw:space': 'cosine'},
)

print(f'Collection \'games\' ready. Documents currently in DB: {collection.count()}')

Collection 'games' ready. Documents currently in DB: 35


In [5]:
# Upsert all documents (safe to re-run — upsert won't duplicate)
collection.upsert(
    documents=all_docs,
    ids=all_ids,
    metadatas=all_meta,
)

print(f'Upsert complete. Documents in collection: {collection.count()}')

Upsert complete. Documents in collection: 35


## 5. Semantic Search Demonstrations

Three queries that test different retrieval scenarios.

In [6]:
def semantic_search(query: str, n_results: int = 3) -> None:
    print(f'Query: "{query}"')
    print('-' * 60)
    results = collection.query(query_texts=[query], n_results=n_results)
    for i, (doc, meta) in enumerate(zip(results['documents'][0], results['metadatas'][0]), 1):
        record_type = meta.get('type', 'unknown')
        name = meta.get('title') or meta.get('name', 'Unknown')
        print(f'  {i}. [{record_type.upper()}] {name}')
        print(f'     {doc[:180]}...')
    print()


# --- Query 1: Game by release year ---
semantic_search('open world RPG released in 2022')

Query: "open world RPG released in 2022"
------------------------------------------------------------


  1. [GAME] Baldur's Gate 3
     Title: Baldur's Gate 3. Developer: Larian Studios. Publisher: Larian Studios. Release date: 2023-08-03. Platforms: PC, PlayStation 5, Xbox Series X/S, macOS. Genre: RPG, Turn-Based...
  2. [GAME] Cyberpunk 2077
     Title: Cyberpunk 2077. Developer: CD Projekt Red. Publisher: CD Projekt. Release date: 2020-12-10. Platforms: PC, PlayStation 4, PlayStation 5, Xbox One, Xbox Series X/S. Genre: Ac...
  3. [GAME] Elden Ring
     Title: Elden Ring. Developer: FromSoftware. Publisher: Bandai Namco Entertainment. Release date: 2022-02-25. Platforms: PC, PlayStation 4, PlayStation 5, Xbox One, Xbox Series X/S....



In [7]:
# --- Query 2: Developer-focused ---
semantic_search('games made by FromSoftware')

Query: "games made by FromSoftware"
------------------------------------------------------------


  1. [COMPANY] FromSoftware
     Company: FromSoftware. Country: Japan. Founded: 1986. Headquarters: Tokyo, Japan. Parent company: Kadokawa Corporation. Current projects: New project in development (unannounced); ...
  2. [GAME] Dark Souls III
     Title: Dark Souls III. Developer: FromSoftware. Publisher: Bandai Namco Entertainment. Release date: 2016-03-24. Platforms: PC, PlayStation 4, Xbox One. Genre: Action RPG, Soulslik...
  3. [GAME] Elden Ring
     Title: Elden Ring. Developer: FromSoftware. Publisher: Bandai Namco Entertainment. Release date: 2022-02-25. Platforms: PC, PlayStation 4, PlayStation 5, Xbox One, Xbox Series X/S....



In [8]:
# --- Query 3: Current projects / future games ---
semantic_search('which studio is working on The Witcher 4')

Query: "which studio is working on The Witcher 4"
------------------------------------------------------------


  1. [COMPANY] CD Projekt Red
     Company: CD Projekt Red. Country: Poland. Founded: 1994. Headquarters: Warsaw, Poland. Parent company: CD Projekt Group. Current projects: The Witcher 4 (codename: Polaris) - in ac...
  2. [GAME] The Witcher 3: Wild Hunt
     Title: The Witcher 3: Wild Hunt. Developer: CD Projekt Red. Publisher: CD Projekt. Release date: 2015-05-19. Platforms: PC, PlayStation 4, Xbox One, Nintendo Switch, PlayStation 5,...
  3. [GAME] Baldur's Gate 3
     Title: Baldur's Gate 3. Developer: Larian Studios. Publisher: Larian Studios. Release date: 2023-08-03. Platforms: PC, PlayStation 5, Xbox Series X/S, macOS. Genre: RPG, Turn-Based...



## 6. Summary

| Step | Result |
|---|---|
| Data loaded | 22 games + 10 companies |
| Documents prepared | 32 text chunks |
| Vector DB | ChromaDB persistent (`./chroma_db`) |
| Embedding model | `text-embedding-3-small` via OpenAI API |
| Semantic search | ✅ Working — 3 queries demonstrated |

The collection is ready to be used by the agent in **Notebook 2**.